# Club 01 — Econometric Analysis in Stata

This notebook documents the **Stata commands used in the empirical analysis**.  
No Python conversion is provided here. The purpose of the notebook is to organize the original Stata code into logical methodological sections with Markdown explanations suitable for a GitHub repository and thesis research documentation.

## Empirical workflow

The commands cover:

1. Panel-data setup and variable transformation
2. Descriptive statistics
3. Fixed-effects estimation
4. Normality diagnostics
5. Cross-sectional dependence
6. Slope heterogeneity
7. Second-generation panel unit-root testing
8. Panel cointegration testing
9. Method-of-Moments Quantile Regression
10. Quantile-regression visualization


## 1. Panel-data setup and variable transformation

This section prepares the dataset for panel-data analysis.

- `encode` converts the country identifier from a categorical/string variable into a numeric panel identifier.
- `xtset` declares the country-year panel structure.
- `gen log_mfp = log(mfp)` creates the natural logarithm of multifactor productivity.

The panel structure is defined by **country** as the cross-sectional unit and **year** as the time dimension.

The logarithmic transformation of MFP is used to transform the dependent variable into natural-log form, which is common in productivity and macroeconomic analysis.

In [ ]:
* Panel-data identification
encode country, gen(country_num)

* Declare country-year panel structure
xtset country_num year

* Natural logarithm of multifactor productivity
gen log_mfp = log(mfp)

## 2. Descriptive statistics

Descriptive statistics provide an initial overview of the variables used in the empirical model.

The `summ` command reports basic distributional information, including the number of observations, mean, standard deviation, minimum, and maximum values.

The variables summarized are the dependent variable `log_mfp` and the explanatory variables `fdi`, `pci`, `er`, `fr`, `pr`, and `gdpp`.

In [ ]:
summ log_mfp fdi pci er fr pr gdpp

## 3. Baseline fixed-effects panel regression

The `xtreg, fe` command estimates a **fixed-effects panel regression**.

The model examines the relationship between multifactor productivity and the explanatory variables while controlling for unobserved, time-invariant country-specific characteristics.

The baseline specification can be represented as:

$$
\ln(MFP_{it}) =
\alpha_i +
\beta_1 FDI_{it} +
\beta_2 PCI_{it} +
\beta_3 ER_{it} +
\beta_4 FR_{it} +
\beta_5 PR_{it} +
\beta_6 GDPP_{it} +
\varepsilon_{it}
$$

where:

- $i$ denotes country;
- $t$ denotes year;
- $\alpha_i$ represents country-specific fixed effects;
- $\beta_1,\ldots,\beta_6$ are the slope coefficients; and
- $\varepsilon_{it}$ is the error term.

The fixed-effects estimator is appropriate when unobserved country characteristics may be correlated with the explanatory variables.

In [ ]:
* Baseline fixed-effects panel regression
xtreg log_mfp fdi pci er fr pr gdpp, fe

## 4. Normality diagnostics

This section examines whether the variables follow a normal distribution.

Two tests are applied:

- **Shapiro–Wilk test (`swilk`)**
- **Shapiro–Francia test (`sfrancia`)**

These are distributional diagnostic tests.

The null hypothesis is that the observations are consistent with a normal distribution. A statistically significant result provides evidence against normality.

The tests are applied to the dependent variable and all explanatory variables.

In [ ]:
* Shapiro-Wilk normality test
swilk log_mfp fdi pci er fr pr gdpp

* Shapiro-Francia normality test
sfrancia log_mfp fdi pci er fr pr gdpp

## 5. Cross-sectional dependence

Cross-sectional dependence refers to the possibility that observations belonging to different countries are correlated with one another.

This issue is important in international and macroeconomic panel datasets because countries may be simultaneously affected by common shocks, international financial conditions, regional developments, global economic cycles, or technological changes.

The `xtcd` command is used to test for cross-sectional dependence.

The variables are divided into two groups in the original analysis:

- `log_mfp`, `fdi`, and `pci`
- `er`, `fr`, `pr`, and `gdpp`

The test is used as a diagnostic before proceeding with panel methods that account for cross-sectional dependence.

In [ ]:
* Cross-sectional dependence tests
xtcd log_mfp fdi pci

xtcd er fr pr gdpp

## 6. Slope heterogeneity test

The `xthst` command is used to examine **slope heterogeneity** across the panel units.

The purpose of the test is to determine whether the estimated relationships are homogeneous across countries or whether the effects of the explanatory variables differ across countries.

The general hypotheses can be expressed as:

$$
H_0: \beta_i = \beta
$$

against the alternative:

$$
H_1: \beta_i \neq \beta
$$

If slope homogeneity is rejected, assuming that every country has exactly the same slope coefficients may be inappropriate. This provides motivation for methods that can accommodate heterogeneous relationships.

In [ ]:
* Test for slope heterogeneity
xthst log_mfp fdi pci er fr pr gdpp

## 7. Second-generation panel unit-root tests: CIPS

The `xtcips` command implements the **cross-sectionally augmented IPS (CIPS) panel unit-root test**.

This is a second-generation panel unit-root approach that is designed to account for cross-sectional dependence.

The analysis first tests the variables in levels. First differences are also tested for variables where the original Stata workflow specifies them.

The general hypotheses are:

$$
H_0: \text{The panel contains a unit root}
$$

against:

$$
H_1: \text{The panel is stationary for at least some cross-sectional units}
$$

The option `maxlags(1)` allows a maximum of one lag, while `bglags(1)` specifies one lag for the cross-sectional augmentation component.

The CIPS tests are important for determining the integration properties of the variables before conducting the panel cointegration analysis.

In [ ]:
* CIPS test for log MFP
xtcips log_mfp, maxlags(1) bglags(1)

* CIPS test for FDI
xtcips fdi, maxlags(1) bglags(1)

* CIPS test for PCI in levels
xtcips pci, maxlags(1) bglags(1)

* CIPS test for first-differenced PCI
xtcips d.pci, maxlags(1) bglags(1)

* CIPS test for ER
xtcips er, maxlags(1) bglags(1)

* CIPS test for FR in levels
xtcips fr, maxlags(1) bglags(1)

* CIPS test for first-differenced FR
xtcips d.fr, maxlags(1) bglags(1)

* CIPS test for PR
xtcips pr, maxlags(1) bglags(1)

* CIPS test for GDPP in levels
xtcips gdpp, maxlags(1) bglags(1)

* CIPS test for first-differenced GDPP
xtcips d.gdpp, maxlags(1) bglags(1)

## 8. Panel cointegration test

The `xtcointtest pedroni` command performs the **Pedroni panel cointegration test**.

The purpose of this test is to determine whether a long-run equilibrium relationship exists among the variables in the empirical model.

The tested relationship is between:

- `log_mfp`
- `fdi`
- `pci`
- `er`
- `fr`
- `pr`
- `gdpp`

The general null hypothesis is:

$$
H_0: \text{There is no cointegration}
$$

Rejection of the null provides evidence of a long-run relationship among the variables.

Cointegration analysis is particularly relevant when the variables are non-stationary but may move together over time.

In [ ]:
* Pedroni panel cointegration test
xtcointtest pedroni log_mfp fdi pci er fr pr gdpp

## 9. Method-of-Moments Quantile Regression (MMQR)

The analysis next applies **Method-of-Moments Quantile Regression (MMQR)** using the `mmqreg` command.

Unlike a conventional mean regression, quantile regression investigates how the explanatory variables are associated with different points of the conditional distribution of multifactor productivity.

The model is estimated at the:

- 25th percentile ($\tau=0.25$)
- 50th percentile / median ($\tau=0.50$)
- 75th percentile ($\tau=0.75$)
- 90th percentile ($\tau=0.90$)

A general quantile specification is:

$$
Q_{\ln(MFP_{it})}(\tau|X_{it})
=
\alpha_{\tau}
+
\beta_{1,\tau}FDI_{it}
+
\beta_{2,\tau}PCI_{it}
+
\beta_{3,\tau}ER_{it}
+
\beta_{4,\tau}FR_{it}
+
\beta_{5,\tau}PR_{it}
+
\beta_{6,\tau}GDPP_{it}
$$

where $\tau$ denotes the selected conditional quantile.

Estimating several quantiles makes it possible to examine whether the effects of the explanatory variables differ between countries/observations located at lower, middle, and higher levels of the conditional productivity distribution.

The MMQR estimates are therefore used to investigate **distributional heterogeneity** in the relationships rather than only the average conditional effect.

In [ ]:
* MMQR at the 25th percentile
mmqreg log_mfp fdi pci er fr pr gdpp, q(25)

* MMQR at the 50th percentile (median)
mmqreg log_mfp fdi pci er fr pr gdpp, q(50)

* MMQR at the 75th percentile
mmqreg log_mfp fdi pci er fr pr gdpp, q(75)

* MMQR at the 90th percentile
mmqreg log_mfp fdi pci er fr pr gdpp, q(90)

## 10. Quantile-regression coefficient plot

The `qregplot` command is used to visualize the estimated quantile-regression coefficients.

The purpose of the plot is to show how the estimated effects change across the conditional distribution.

The graphical analysis helps identify:

- whether coefficients remain stable across quantiles;
- whether the magnitude of an effect increases or decreases;
- whether the sign of an effect changes across quantiles; and
- whether the relationship between the explanatory variables and productivity is heterogeneous across the productivity distribution.

This provides a visual complement to the numerical MMQR results.

In [ ]:
* Plot quantile-regression coefficients
qregplot

## 11. Summary of the econometric methodology

The complete empirical strategy follows a sequential panel-data framework:

**Panel setup → Descriptive statistics → Fixed effects → Normality diagnostics → Cross-sectional dependence → Slope heterogeneity → CIPS unit-root tests → Pedroni cointegration → MMQR → Quantile coefficient plot**

This sequence first establishes the structure and characteristics of the panel dataset, then evaluates important panel-data properties, examines long-run relationships, and finally estimates heterogeneous effects across different points of the conditional productivity distribution.

The Stata commands in this notebook are the original commands used in the empirical analysis; the Markdown sections provide methodological documentation for reproducibility and GitHub presentation.